# Drawable Colab runtime preflight

Checks only the manually selected hosted runtime. It does not read/upload training bundles, download/load models, test model inference, create an optimizer, run backward passes, write checkpoints, or modify weights. A successful tensor smoke test proves only that this temporary VM can execute a tiny tensor operation. Colab availability/session lifetime varies; keep this as an interactive preflight, not a background worker. Save the notebook with outputs cleared. All authorization/transfer/training flags remain false.

In [ ]:
from datetime import datetime, timezone
import importlib.metadata
import importlib.util
import json
import os
import platform
import shutil
report = {'schema_version': 2, 'profile': 'colab-cuda-runtime-preflight', 'checked_at': datetime.now(timezone.utc).isoformat(), 'python': platform.python_version(), 'platform': platform.platform(), 'colab_release_tag': os.environ.get('COLAB_RELEASE_TAG'), 'gpu': {'status': 'not_tested'}, 'models_loaded': False, 'inference_tested': False, 'backward_feasibility_tested': False, 'optimizer_created': False, 'training_authorized': False, 'training_started': False, 'data_uploaded': False, 'network_used': False}
for package in ('torch', 'torchvision', 'timm', 'open_clip_torch', 'transformers', 'Pillow', 'pydantic', 'safetensors'):
    try: report.setdefault('packages', {})[package] = importlib.metadata.version(package)
    except importlib.metadata.PackageNotFoundError: report.setdefault('packages', {})[package] = 'not-installed'
disk = shutil.disk_usage('/content' if os.path.isdir('/content') else '.')
report['runtime_disk'] = {'free_bytes': disk.free, 'total_bytes': disk.total}
py = tuple(int(part) for part in platform.python_version_tuple()[:2])
report['python_311_compatible'] = py == (3, 11)
if importlib.util.find_spec('torch') is not None:
    import torch
    report['torch'] = {'version': torch.__version__, 'cuda_build': torch.version.cuda, 'cudnn': torch.backends.cudnn.version()}
    if torch.cuda.is_available():
        try:
            device = torch.device('cuda:0')
            result = (torch.ones((32, 32), device=device) @ torch.ones((32, 32), device=device)).sum().item()
            torch.cuda.synchronize(device)
            properties = torch.cuda.get_device_properties(device)
            free_memory, total_memory = torch.cuda.mem_get_info(device)
            report['gpu'] = {'status': 'available', 'name': properties.name, 'compute_capability': f'{properties.major}.{properties.minor}', 'total_memory_bytes': total_memory, 'free_memory_bytes': free_memory, 'tensor_smoke_result': result, 'pytorch_cuda': torch.version.cuda}
        except Exception as error: report['gpu'] = {'status': 'probe_failed', 'error_type': type(error).__name__}
    else: report['gpu'] = {'status': 'unavailable_in_this_runtime'}
else: report['gpu'] = {'status': 'not_tested', 'reason': 'PyTorch is not installed; this notebook does not install dependencies.'}
print(json.dumps(report, indent=2, sort_keys=True))
with open('/content/drawable-colab-preflight.json', 'w', encoding='utf-8') as output: json.dump(report, output, indent=2, sort_keys=True)
print('Local report: /content/drawable-colab-preflight.json')